In [67]:
import pandas as pd
import itertools

In [68]:
validation_set = pd.read_csv('/home/s6moakba/Thesis/SEM14_res_val_performance.csv')

training_set = pd.read_csv('/home/s6moakba/InstructABSA/Dataset/SemEval14/Train/Restaurants_Train.csv')

In [69]:
training_set.shape

(3041, 4)

In [70]:
training_set['aspectTerms'] = training_set['aspectTerms'].apply(eval)

In [72]:
def extract_terms(aspect_terms_list):
    terms = []
    for aspect in aspect_terms_list:
        terms.append(aspect['term'])
    return terms

training_set['terms'] = training_set['aspectTerms'].apply(extract_terms)

In [73]:
def replace_terms_with_blanks(row):
    pr_text = row['raw_text']
    for term in row['terms']:
        pr_text = pr_text.replace(term, '[MASK]')
    return pr_text
training_set['text_blank'] = training_set.apply(replace_terms_with_blanks, axis=1)


In [74]:
training_set_filtered = training_set[training_set['terms'].apply(lambda x: x != ['noaspectterm'])]
training_set_filtered

,sentenceId,raw_text,aspectTerms,aspectCategories,terms,text_blank
0,3121,But the staff was so horrible to us.,"[{'term': 'staff', 'polarity': 'negative'}]","[{'category': 'service', 'polarity': 'negative'}]",[staff],But the [MASK] was so horrible to us.
1,2777,"To be completely fair, the only redeeming fact...","[{'term': 'food', 'polarity': 'positive'}]","[{'category': 'food', 'polarity': 'positive'},...",[food],"To be completely fair, the only redeeming fact..."
2,1634,"The food is uniformly exceptional, with a very...","[{'term': 'food', 'polarity': 'positive'}, {'t...","[{'category': 'food', 'polarity': 'positive'}]","[food, kitchen, menu]","The [MASK] is uniformly exceptional, with a ve..."
5,2846,"Not only was the food outstanding, but the lit...","[{'term': 'food', 'polarity': 'positive'}, {'t...","[{'category': 'food', 'polarity': 'positive'},...","[food, perks]","Not only was the [MASK] outstanding, but the l..."
7,1458,Our agreed favorite is the orrechiete with sau...,[{'term': 'orrechiete with sausage and chicken...,"[{'category': 'food', 'polarity': 'positive'},...","[orrechiete with sausage and chicken, waiters,...",Our agreed favorite is the [MASK] (usually the...
...,...,...,...,...,...,...
3033,2378,"The service was typical short-order, dinner type.","[{'term': 'service', 'polarity': 'neutral'}]","[{'category': 'service', 'polarity': 'neutral'}]",[service],"The [MASK] was typical short-order, dinner type."
3034,1027,"We shared a bottle of sake, an order of edamam...","[{'term': 'bottle of sake', 'polarity': 'neutr...","[{'category': 'food', 'polarity': 'neutral'}]","[bottle of sake, edamames, sushi plate, sashimi]","We shared a [MASK], an order of [MASK], and sh..."
3035,1735,I can't believe people complain about no chees...,"[{'term': 'cheese sticks', 'polarity': 'neutra...","[{'category': 'anecdotes/miscellaneous', 'pola...",[cheese sticks],I can't believe people complain about no [MASK]?
3037,777,"From the appetizers we ate, the dim sum and ot...","[{'term': 'appetizers', 'polarity': 'positive'...","[{'category': 'food', 'polarity': 'positive'}]","[appetizers, dim sum, foods, food]","From the [MASK] we ate, the [MASK] and other v..."


In [75]:
row = training_set_filtered.iloc[20]
print(row['raw_text'])
print(row['text_blank'])


Great food at REASONABLE prices, makes for an evening that can't be beat!
Great [MASK] at REASONABLE [MASK], makes for an evening that can't be beat!


In [76]:
terms_lengths = training_set_filtered['terms'].apply(len).tolist()
mask_counts = training_set_filtered['text_blank'].str.count('\[MASK\]').tolist()
print(terms_lengths)
print(mask_counts)

# Find indices where the values in the two lists are different
different_indices = [i for i, (x, y) in enumerate(zip(terms_lengths, mask_counts)) if x != y]
print(different_indices)

[1, 1, 3, 2, 4, 1, 1, 7, 2, 2, 1, 2, 2, 1, 1, 2, 2, 3, 1, 3, 2, 2, 1, 2, 1, 2, 1, 5, 1, 2, 1, 2, 1, 1, 1, 1, 1, 3, 3, 3, 4, 1, 1, 4, 2, 1, 2, 1, 2, 2, 1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2, 1, 3, 1, 2, 3, 2, 1, 2, 2, 1, 1, 1, 3, 1, 2, 2, 1, 6, 3, 1, 1, 1, 1, 1, 3, 2, 1, 1, 1, 1, 1, 2, 3, 1, 2, 1, 2, 1, 3, 4, 2, 1, 1, 2, 3, 3, 2, 1, 3, 1, 1, 1, 1, 2, 3, 2, 2, 2, 2, 1, 2, 2, 4, 2, 2, 3, 2, 7, 1, 1, 1, 2, 1, 2, 1, 1, 3, 1, 1, 1, 2, 1, 1, 1, 1, 2, 5, 1, 3, 2, 2, 1, 1, 1, 1, 1, 1, 2, 1, 3, 3, 3, 2, 8, 3, 2, 1, 1, 3, 1, 1, 1, 1, 4, 1, 3, 2, 2, 1, 4, 1, 1, 1, 2, 1, 4, 1, 1, 1, 3, 2, 2, 3, 1, 1, 3, 1, 1, 2, 1, 2, 3, 1, 3, 1, 1, 4, 2, 1, 1, 2, 4, 2, 2, 1, 1, 1, 1, 3, 1, 1, 1, 1, 3, 2, 1, 1, 3, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 3, 1, 5, 2, 4, 3, 3, 3, 2, 1, 1, 1, 1, 2, 1, 1, 1, 3, 1, 2, 1, 2, 1, 2, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 2, 1, 1, 1, 1, 4, 1, 1, 1, 3, 1, 1, 4, 1, 2, 1, 1, 3, 1, 2, 2, 2, 1, 1, 1, 3, 3, 3, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 1, 1, 2, 2, 2, 2, 1, 3, 3, 1, 1, 4, 1, 2, 

<>:2: SyntaxWarning: invalid escape sequence '\['
<>:2: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_1192107/2894125903.py:2: SyntaxWarning: invalid escape sequence '\['
  mask_counts = training_set_filtered['text_blank'].str.count('\[MASK\]').tolist()


In [77]:
row = training_set_filtered.iloc[different_indices[10]]
print(row['terms'])
print(row['raw_text'])
print(row['text_blank'])

IndexError: list index out of range

In [47]:
training_set_filtered.iloc[different_indices[9]]['text_blank'] = "I've had to wait only a few times during [MASK] but this place is definitely worth the [MASK]."

/tmp/ipykernel_1192107/4238507440.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  training_set_filtered.iloc[different_indices[9]]['text_blank'] = "I've had to wait only a few times during [MASK] but this place is definitely worth the [MASK]."


In [78]:
test_prompt = """
Your task is to replace the [MASK] in the following sentence with the best-fitting words. Follow these instructions carefully:
- Domain: Restaurant reviews.
- Ensure the output contains the same number of words as there are blanks in the input sentence.
- Maintain the same order of suggestions as the blanks appear in the input.
- Use **double quotes ("example")** around each blank suggestion.
- Put the suggestions in a list.
- Avoid repeating input text in the output.
- Try to include synonyms where appropriate
- ONLY OUTPUT SUGGESTIONS, NOTHING ELSE
- Good examples:
### input ###
sentence:  [MASK] are good and still rare to find in NYC.
### Output ### 
["Deep Fried Skewers"]
### input ###
sentence: And Kruno, the [MASK] is the best [MASK] I have yet to come across.
### Output ### 
["beverage manager", "bartender"]
### input ###
sentence: the [MASK] at Ping's are clear as glass with healthy-looking creatures who do not yet know that they will be part of some [MASK] lover's [MASK].
### Output ### 
["tanks", "tanks", "dim sum", "brunch"]


Now complete this task like examples above:
### Input ###"""

In [79]:
prompt_begining =  """
Your task is to replace the [MASK] in the following sentence with the best-fitting words. Follow these instructions carefully:
- Domain: Restaurant reviews.
- Ensure the output contains the same number of words as there are blanks in the input sentence.
- Maintain the same order of suggestions as the blanks appear in the input.
- Use **double quotes ("example")** around each blank suggestion.
- Put the suggestions in a list.
- Avoid repeating input text in the output.
- Try to include synonyms where appropriate
- ONLY OUTPUT SUGGESTIONS, NOTHING ELSE
"""

In [80]:
additional_instruction = [
        "-Think creatively about this sentence.",
        "-Consider alternative ways to complete the blanks.",
        "-Suggest unique words to fit the blanks.",
    ]

In [81]:
examples_len_1 = [
"""
### input ###
sentence: The [MASK] have an outstanding taste with a terrific texture, both chewy yet not gummy.
### Output ### 
["Bagels"]
""" , 
"""
### input ###
sentence: He has visited Thailand and is quite expert on the cuisine.
### Output ###
["cuisine"]
""",
"""
### input ###
sentence: I asked for seltzer with lime, no ice.
### Output ###
["seltzer with lime"]
"""]

In [82]:
examples_len_3 = [
"""
### input ###
sentence:  The [MASK] is uniformly exceptional, with a very capable [MASK] which will proudly whip up whatever you feel like eating, whether it's on the [MASK] or not.
### Output ### 
["food", "kitchen", "menu"]
""" , 
"""
### input ###
sentence: Yes, they use fancy [MASK], but even fancy [MASK] don't make for good [MASK] unless someone knows how to get the [MASK] right.
### Output ###
["ingredients", "ingredients", "pizza", "crust"]
""",
"""
### input ###
sentence: A narrow [MASK] leads to a tiny [MASK] where there are three tiny white tiled [MASK], a great deal of mess (stacks of bottles, cans) and a small [MASK] holding 12-14 [MASK].
### Output ###
["corridor", "space", "counters", "counter", "entrees"]
""",
"""
### input ###
sentence: The have over 100 different [MASK] to offer thier guest so that made my husband very happy and the [MASK] was delicious, if I must recommend a [MASK] it must be the [MASK].
### Output ###
["beers", "food", "pumkin tortelini", "dish"]
""",
"""
### input ###
sentence: The [MASK] was impeccable and unobtrusive -- the [MASK] knows what they are there to do -- to know their [MASK], present your [MASK], and attend to your needs.
### Output ###
["service", "staff", "menu", "meal"]
"""]

In [83]:
examples_len_2 = [
"""
### input ###
sentence:  I stumbled upon this second floor walk-up two Fridays ago when I was with two friends in town from L.A. Being serious [MASK] lovers, we sat at the [MASK] to be closer to the action.
### Output ### 
["sushi", "sushi bar"]
""" , 
"""
### input ###
sentence: The [MASK] is the best if you like [MASK].
### Output ###
["pizza", "thin crusted pizza"]
""",
"""
### input ###
sentence: All the money went into the [MASK], none of it went to the [MASK].
### Output ###
["interior decoration", "chefs"]
""",
"""
### input ###
sentence: Don't go alone---even two people isn't enough for the whole experience, with [MASK] and a [MASK].
### Output ###
["pickles", "selection of meats and seafoods"]
""",
"""
### input ###
sentence: The [MASK] was really good, I had the [MASK] and it was one of the best ever.
### Output ###
["food", "onion soup"]
"""]

In [84]:
import random
def create_prompt():
    prompt = prompt_begining
    prompt  += "\n" + random.choice(additional_instruction)
    prompt += "\n" + "-Good examples:"
    prompt += "\n" + random.choice(examples_len_1)
    prompt += "\n" + random.choice(examples_len_2)
    prompt += "\n" + random.choice(examples_len_3) 
    prompt += "\n" +"""Now complete this task like examples above: \n### Input ###"""
    return prompt

In [85]:

def preprocess_data(df,iter):
    data = []
    for _, row in df.iterrows():
        for i  in range(iter):
            blanked_text = row['text_blank']
            prompt = create_prompt() + f"\nsentence: {blanked_text}\n###Output### "
            data.append({
                'blank': blanked_text,
                'original_terms': row['terms'],
                'original_text': row['raw_text'],
                'prompt': prompt,
                'aspectTerms_old': row['aspectTerms'],
            })
    return pd.DataFrame(data)

In [86]:
processed_train = preprocess_data(training_set_filtered, 6)

In [87]:
processed_train.shape

(12126, 5)

In [59]:
row = processed_train.iloc[5]
print(row['prompt'])
# print(row['original_terms'])


Your task is to replace the [MASK] in the following sentence with the best-fitting words. Follow these instructions carefully:
- Domain: Restaurant reviews.
- Ensure the output contains the same number of words as there are blanks in the input sentence.
- Maintain the same order of suggestions as the blanks appear in the input.
- Use **double quotes ("example")** around each blank suggestion.
- Put the suggestions in a list.
- Avoid repeating input text in the output.
- Try to include synonyms where appropriate
- ONLY OUTPUT SUGGESTIONS, NOTHING ELSE

-Suggest unique words to fit the blanks.
-Good examples:

### input ###
sentence: He has visited Thailand and is quite expert on the cuisine.
### Output ###
["cuisine"]


### input ###
sentence: Don't go alone---even two people isn't enough for the whole experience, with [MASK] and a [MASK].
### Output ###
["pickles", "selection of meats and seafoods"]


### input ###
sentence: The [MASK] was impeccable and unobtrusive -- the [MASK] know

In [60]:
processed_train.shape

(12126, 5)

In [61]:
processed_train.to_csv('/home/s6moakba/train_res_14_dynamic_sample.csv')

# Check results
